In [ ]:
from utils import *
import seaborn as sns
from scipy import stats
from sklearn.metrics import r2_score , mean_squared_error
import torch

In [66]:
def confidence_interval(x, y): # x and y are np arrays
    # Perform linear regression with numpy.polyfit and return covariance matrix
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    # Number of data points
    n = len(x)

    # Degrees of freedom: n - 2 (because we estimated two parameters: slope and intercept)
    dof = n - 2

    # t-critical value for 95% confidence interval
    t_critical = stats.t.ppf(1 - 0.005, dof)

    # Calculate confidence intervals
    slope_CI_lower = slope - t_critical * std_err
    slope_CI_upper = slope + t_critical * std_err
    
    return slope, intercept, r_value, p_value, std_err, slope_CI_lower, slope_CI_upper

def fit_IFE(df):    
    data1 = pd.DataFrame({'x': df['pd_bias'],
                        'y': df['PD-PD'],
                        'label': df['prime_verb'],
                        'structure': 'PD-PD'})

    data2 = pd.DataFrame({'x': df['pd_bias'],
                        'y': df['DO-PD'],
                        'label': df['prime_verb'],
                        'structure': 'DO-PD'})
    
    result = {'PD-PD_slope':-100, 'PD-PD_intercept':-100, 'PD-PD_R2':-100, 'PD-PD_RMSE':-100, 'DO-PD_slope':-100, 'DO-PD_intercept':-100, 'DO-PD_R2':-100, 'DO-PD_RMSE':-100}

    # Plot each scatter plot with regression line and label each point
    for data in [data1, data2]:
        # Fit a line (perform linear regression) and calculate R-squared and RMSE
        coefficients = np.polyfit(df['pd_bias'], data['y'], 1)
        poly = np.poly1d(coefficients)
        y_pred = poly(df['pd_bias']) # Calculate the predicted values (y_pred)
        r_squared = r2_score(data['y'], y_pred) # Calculate R-squared value
        rmse = np.sqrt(mean_squared_error(data['y'], y_pred)) # Calculate Root Mean Squared Error (RMSE)
        
        # Find std_err and 95% confidence interval
        slope, intercept, r_value, p_value, std_err, slope_CI_lower, slope_CI_upper = confidence_interval(df['pd_bias'].to_numpy(), data['y'].to_numpy())
        
        # record relevant info
        result[f"{data['structure'].tolist()[0]}_slope"] = slope
        result[f"{data['structure'].tolist()[0]}_intercept"] = intercept
        result[f"{data['structure'].tolist()[0]}_std_err"] = std_err
        result[f"{data['structure'].tolist()[0]}_CI_lower"] = slope_CI_lower
        result[f"{data['structure'].tolist()[0]}_CI_upper"] = slope_CI_upper
        result[f"{data['structure'].tolist()[0]}_r_value"] = r_value
        result[f"{data['structure'].tolist()[0]}_p_value"] = p_value
        result[f"{data['structure'].tolist()[0]}_R2"] = r_squared
        result[f"{data['structure'].tolist()[0]}_RMSE"] = rmse

    return result

In [70]:
MODEL = 'FT'
if MODEL == 'GPT2':
    sizes = ['small', 'medium', 'large']
elif MODEL == 'GPT3':
    sizes = ['davinci-002']
elif MODEL == 'Llama':
    sizes = ['7b', '7b-chat', '13b']
elif MODEL == 'FT':
    sizes = ['NotSquared', 'Squared']
else:
    raise ValueError('Invalid MODEL')

intermediate_path = f'{ROOT_DIR}/results/{MODEL}/'
data = pd.read_csv(intermediate_path+f'IFE_{MODEL}.csv')
temp = []
col_names = ['size', 'pronoun',
            'PD-PD_slope', 'PD-PD_intercept', 'PD-PD_std_err', 'PD-PD_CI_lower', 'PD-PD_CI_upper', 'PD-PD_r_value', 'PD-PD_p_value', 'PD-PD_R2', 'PD-PD_RMSE',
            'DO-PD_slope', 'DO-PD_intercept', 'DO-PD_std_err', 'DO-PD_CI_lower', 'DO-PD_CI_upper', 'DO-PD_r_value', 'DO-PD_p_value', 'DO-PD_R2', 'DO-PD_RMSE']
df_IFE = pd.DataFrame(temp, columns=col_names)
    
for SIZE in sizes:
    for PRONOUN in [True, False]:
        if MODEL == 'FT':
            str_sq = True if SIZE == 'Squared' else False
            data_sub = data[(data['squared'] == str_sq) & (data['pronoun'] == PRONOUN)]
        else:
            data_sub = data[(data['size'] == SIZE) & (data['pronoun'] == PRONOUN)]
        dic = {'size':SIZE, 'pronoun':PRONOUN}
        result_dic = fit_IFE(data_sub)
        merged_dic = {**dic, **result_dic}
        merged_list = {key: [value] for key, value in merged_dic.items()}
        df_IFE = pd.concat([df_IFE, pd.DataFrame(merged_list)], ignore_index=True)
            
df_IFE.to_csv(intermediate_path+f'IFE_{MODEL}_Stats.csv', index=False)

/var/folders/rd/jz160lxn3g5bc3cyzjv46lmr0000gp/T/ipykernel_39523/1198374961.py:32: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_IFE = pd.concat([df_IFE, pd.DataFrame(merged_list)], ignore_index=True)
/var/folders/rd/jz160lxn3g5bc3cyzjv46lmr0000gp/T/ipykernel_39523/1198374961.py:32: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_IFE = pd.concat([df_IFE, pd.DataFrame(merged_list)], ignore_index=True)
/var/folders/rd/jz160lxn3g5bc3cyzjv46lmr0000gp/T/ipykernel_39523/1198374961.py:32: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_IFE = pd.concat([df_IFE, pd.DataFrame(merged_list)], ignore_index=True)
